# Reproducibility — inspect a manifest

1. Load a raw JSON manifest from `research/captures/`.
2. Inspect its `runConfig` and `repeat_group`.
3. Understand `manifestVersion` forward-compat semantics.
4. Exercise `self_eval_guard` to surface the self-bias warning.

Paths resolve via a `go.mod` walk-up so this works from both the tracked location and `.reporting/notebooks/`.

In [ ]:
from pathlib import Path
import json
import os


def _repo_root():
    here = Path(os.getcwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "go.mod").exists():
            return p
    raise SystemExit("repo root (go.mod) not found — run via scripts/report.sh")


ROOT = _repo_root()
CAPTURES_ROOT = ROOT / "research" / "captures"
DB_PATH = ROOT / "reports" / "resolver.duckdb"

In [ ]:
manifest_path = None
if CAPTURES_ROOT.exists():
    for manifests_dir in sorted(CAPTURES_ROOT.glob("*/*/manifests")):
        run_dir = manifests_dir.parent
        if (run_dir / "scorecard.json").exists():
            candidates = sorted(manifests_dir.glob("*.json"))
            if candidates:
                manifest_path = candidates[0]
                break

if manifest_path is not None:
    print(f"using manifest: {manifest_path.relative_to(ROOT)}")
    m = json.loads(manifest_path.read_text())
else:
    print("no manifests under research/captures/ — using synthetic fallback.")
    m = {
        "manifestVersion": 2,
        "runId": "synthetic-001",
        "model": "gresh-general",
        "tier": "1",
        "runConfig": {
            "virtualModel": "gresh-general",
            "realModel": "SomeOrg/SomeModel",
            "repeatGroup": "group-alpha",
            "notes": "synthetic fallback for notebook demo",
        },
    }

## runConfig and repeat_group

`repeatGroup` ties runs from the same `--n N` invocation so they can be compared statistically.

In [ ]:
run_config = m.get("runConfig") or {}
print("runConfig:")
print(json.dumps(run_config, indent=2, default=str))

repeat_group = run_config.get("repeatGroup") or run_config.get("repeat_group")
print(f"\nrepeat_group: {repeat_group!r}")

## manifestVersion forward-compat

- `1` — original schema (pre-runConfig embedding).
- `2` — current schema; `runConfig` embedded.
- `>2` — forward-compat: Go ingest warns and continues, silently ignoring unknown fields.

In [ ]:
CURRENT_SCHEMA_VERSION = 2
manifest_version = m.get("manifestVersion", 1)
print(f"manifestVersion in file: {manifest_version}")
print(f"current SchemaVersion  : {CURRENT_SCHEMA_VERSION}")
if manifest_version > CURRENT_SCHEMA_VERSION:
    print(
        f"WARN: manifestVersion={manifest_version} > {CURRENT_SCHEMA_VERSION}. "
        "Go ingest emits a forward-compat warning for this file."
    )
else:
    print("schema version is current — no forward-compat warning.")

## self_eval_guard

If the reporter LLM is also a model under test, the benchmark becomes self-grading. `self_eval_guard` emits a stderr warning.

In [ ]:
import io
import sys

if not DB_PATH.exists():
    print(f"DuckDB missing at {DB_PATH} — run scripts/report.sh first.")
else:
    from analyze.report import self_eval_guard
    from analyze.db import Store

    with Store(DB_PATH) as s:
        runs = s.run_summaries()

    reporter = runs[0].model if runs else "gresh-general"
    print(f"calling self_eval_guard with reporter_model={reporter!r}...")

    buf = io.StringIO()
    old_stderr, sys.stderr = sys.stderr, buf
    try:
        self_eval_guard(runs, reporter)
    finally:
        sys.stderr = old_stderr

    captured = buf.getvalue()
    if captured:
        print("stderr output:")
        print(captured)
    else:
        print("no warning emitted (reporter is not in the tested model set).")

## Done

- See **`quickstart.ipynb`** for the dashboard view.
- Re-run `scripts/report.sh` after adding captures.